# Data cataloguing for Xarray

**Goals:** At the end of this tutorial, you'll have an overview of what data cataloging is, why it is done, what tools are available. You'll also know how to open and browse some data catalogues.

> A data catalog is a detailed inventory of data assets within an organization. It helps users easily discover, understand, manage, curate and access data. -[IBM](https://www.ibm.com/think/topics/data-catalog)

Within an organisation, you may work with a lot of different datasets. As someone managing this data, you might want the following things:

- **An organized, easily browsable collection**: By easing discovery and loading of your datasets you can reduce the friction for those analysing the data. Users can eaily get an overview of available datasets, discover datasets of interest, and load them for analysis.
- **Access control and logging**: You can limit who is able to access these datasets, and track when datasets are being accessed - keeping the data private, and providing metrics that can inform data management (e.g., by removing little-used datasets).
- **Combine individual data assets**: Combine individual data assets that are similar into a larger object (e.g., combining individual time snapshots into a larger data cube)

There are a bunch of solutions available in Xarray that meet some of these needs - we'll broadly call these "data catalogs". These solutions differ in functionality, and the correct data cataloguing solution depends on your needs as an institute.

If you feel there are data cataloguing tools missing from this page, please submit a PR.

## intake v2

Intake is a lightweight Python package which allows one to specify data catalogues (or a hierarchy of catalogues) via YAML files. These YAML files describe:

- how to load the dataset. This is done via "readers", which support different file formats, and support lazily defining additional transformations to the dataset using Dask
- additional metadata for the dataset

Uploading the catalogue file to a Git forge like GitHub allows for versioning of this file.

This catalogue can be browsed in Python, or via third-party tools which parse the datalog, allowing the user to select and open a dataset. This tool has been adopted in the geosciences community for working with Xarray datasets, but is also flexible to work with other dataset types.

There is no separate authentication layer in Intake, hence access control and logging depends entirely on the way the data is accessed. If the dataset is loaded from a file location on a mounted drive, then you need to have access to that mounted drive. If the dataset is located in an S3 bucket, you need to have the appropriate access credentials for the resource.

Intake was originally developed within Anaconda by the team behind Dask.

### Resources

Project Pythia has a [cookbook on Intake](https://projectpythia.org/intake-cookbook/notebooks/creating-catalogs/).

### Example code

Here we work based off [an Intake tutorial on easy.gems!](https://easy.gems.dkrz.de/Processing/intake.html) from the DKRZ (German Climate Supercomputing Center). Note that the tutorial is based on v1 of Intake (which also uses `intake-xarray`).


In [ ]:
# Browse an existing catalogue

import intake

cat = intake.open_catalog("https://data.nextgems-h2020.eu/online.yaml")
# TIP: Its just a YAML file - open it in your browser and you'll see!

# view the available subcatalogues or datasets
list(cat)

In [ ]:
# Access catalogue elements using `["attr"]` or `.attr` notation
# NOTE: The `.attr` notation only works if the entry is a valid Python attribute name - which the entry might not be

# Let's see what the `tutorial` item is...
cat["tutorial"]

In [ ]:
# it's a subcatalogue!

print(list(cat["tutorial"]))

# Let's look at an entry within...
cat['tutorial']['ICON.native.2d_PT6H_inst']

In [ ]:
# Let's open it as an xarray dataset

ds = cat['tutorial']['ICON.native.2d_PT6H_inst'].to_dask()
ds

That's it for the example! Feel free to poke around further to see what other datasets are available in this catalogue, or to see which other catalogues are available online.

## STAC

> SpatioTemporal Asset Catalogs
> The STAC specification is a common language to describe geospatial information, so it can more easily be worked with, indexed, and discovered.
> -stacspec.org

Let's set the scene a bit: During satellite data processing, many image files (predominantly TIFF) get generated. Each of these files can have different spatial and temporal extents, and over time these individual files can number in the millions, containing terrabytes of data in total. When doing analyses on these files, the first thing one needs to know is whether the file is even relevant to the analysis at hand (i.e., at least "what is the spatial and temporal extent of the file?"), however downloading the file just to check this metadata within is very wasteful. This is the context from which the STAC project was born.

The goal of the STAC project is to make geospatial data more accessible by providing a specification to describe assets and their metadata. Adding this metadata alongside the asset (or lifting it out above the asset itself) allows:

- The efficient querying of assets based on properties of interest without downloading the entire asset.
- The combining of these assets into a datacube object (i.e., Xarray dataset) by reprojecting or interpolating the data to a common grid.


Within the STAC project is:

- **The STAC Specification**: A set of specifications describing (from most atomic to least) STAC Items, STAC Catalogs, and STAC collections. There is also a spec for the STAC API.
- **Tooling for working with STAC**: More on this just now
- **The community**

There are various tools in the STAC ecosystem. There are online STAC browsers such as the [Radiant Earth STAC browser](https://radiantearth.github.io/stac-browser/#/). There are Python packages for the creating of STAC items and searching of STAC catalogues (e.g., [`rustac-py`](https://stac-utils.github.io/rustac/) and [`pystac`](https://github.com/stac-utils/pystac)). There are also Python packages for combining STAC objects into larger datacubes (e.g., [`odc-stac`](https://github.com/opendatacube/odc-stac), [`stacstac`](https://github.com/gjoseph92/stackstac) and [`xpystac`](https://github.com/stac-utils/xpystac)).

There are many packages with some overlapping functionality. An example workflow you might follow would be using an online STAC browser to get a better understanding of the STAC assets you're working with, and what properties are available to query on, using `rustac-py` to generate and run the query, and then (once you're happy with the result) using `stackstac` or `odc-stac` to open the result as an Xarray Dataset.

### More resources

- https://stacspec.org/en
- https://stacindex.org/

### Example using odc-stac



## Commercial offering - Arraylake

There are also commercial offerings available that have data cataloguing functionality.

Arraylake is Earthmover's managed cloud platform built on top of the open-source Icechunk data format
(where Icechunk is a project offering versioning of Zarr-like datasets).

See [here](https://icechunk.io/en/latest/arraylake/) for a full comparison between Icechunk and Arraylake.

A notable feature is that Arraylake integrates with [Earthmover's data marketplace](https://www.earthmover.io/marketplace/),
powering data discovery and allowing for the dissemination of datasets (or subsets of datasets) to other interested users.



## More resources

https://guide.cloudnativegeo.org/cookbooks/zarr-stac-report/data-consumers/